# Анализ датасета

В качестве исходных данных для лабораторной работы выбран датасет Credit Card Fraud Detection

Данный датасет был выбран по нескольким ключевым причинам:
- Во-первых, его объем достаточен для демонстрации преимуществ распределенной обработки в Hadoop/Spark по сравнению с локальными вычислениями, но при этом позволяет быстро итерировать разработку;
- Во-вторых, разнообразие типов данных и наличие геопространственных и временных атрибутов позволяет реализовать комплексный аналитический пайплайн с операциями агрегации, оконными функциями, гео-расчетами и join-операциями, что идеально подходит для тестирования оптимизаций параллелизма и кэширования;
- В-третьих, несмотря на наличие целевой переменной is_fraud, мы намеренно исключаем ее из анализа, формулируя задачу как неконтролируемую поведенческую аналитику (unsupervised behavioral analytics), что делает пайплайн более универсальным и приближенным к реальным сценариям разведочного анализа данных в финансовой сфере

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [25]:
df = pd.read_csv('train.csv')

print(f"Размер датасета: {df.shape[0]:,} строк × {df.shape[1]} столбцов")

Размер датасета: 1,296,675 строк × 23 столбцов


In [26]:
display(df.head())
print("Типы данных:")
print(df.dtypes.value_counts())

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,city,state,zip,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,0b242abb623afc578575680df30655b9,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1f76529f8574734946361c461b024d99,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,a1a22d70485983eac12b5b88dad1cf95,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,6b849c168bdad6f867558c3793159a81,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,a41d7549acf90789359a9aa5346dcb46,1325376186,38.674999,-78.632459,0


Типы данных:
object     12
int64       6
float64     5
Name: count, dtype: int64


In [27]:
print("Пропущенные значения:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "Нет пропусков")

Пропущенные значения:
Нет пропусков


Оригинальный датасет содержит 1,296,675 объектов и 23 признака (включая целевую переменную is_fraud), среди которых 12 признаков - категориальные



---



## Прототип пайплайна

Цель: построить аналитический конвейер для сегментации пользователей банковских карт на основе их поведенческих паттернов. Пайплайн выявляет пользователей с нетипичным поведением (высокая активность, крупные траты, географическая мобильность, разнообразие категорий) и проводит сравнительный анализ их транзакций с общей популяцией

Пайплайн работает исключительно с признаками транзакций и не использует метку is_fraud. Это имитирует реальную задачу разведочного анализа, когда аномалии ищутся до обучения моделей

In [28]:
def preprocess_data(df):
    """Базовая предобработка: временные признаки, категории, гео-признаки"""

    # Временные признаки
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['hour'] = df['trans_date_trans_time'].dt.hour
    df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['month'] = df['trans_date_trans_time'].dt.month

    # Биннинг сумм транзакций (для анализа)
    df['amount_bin'] = pd.cut(df['amt'],
                              bins=[0, 10, 50, 100, 500, np.inf],
                              labels=['tiny', 'small', 'medium', 'large', 'huge'])

    # Категориальные признаки → category для экономии памяти
    cat_cols = ['category', 'gender', 'state', 'job']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Гео-признаки: расстояние между клиентом и мерчантом (упрощённо)
    geo_cols = ['lat', 'long', 'merch_lat', 'merch_long']
    if all(col in df.columns for col in geo_cols):
        df['geo_distance_km'] = np.sqrt(
            (df['lat'] - df['merch_lat'])**2 +
            (df['long'] - df['merch_long'])**2
        ) * 111  # приблизительный перевод в км

    return df

df = preprocess_data(df)
print("Предобработка завершена")

Предобработка завершена


In [30]:
def compute_basic_stats(df):
    """
    Вычисляет базовую статистику без использования целевой переменной.
    """
    # Статистика по категориям
    category_stats = df.groupby('category').agg(
        total_transactions=('trans_num', 'count'),
        unique_users=('cc_num', 'nunique'),
        avg_amount=('amt', 'mean'),
        median_amount=('amt', 'median'),
        std_amount=('amt', 'std'),
        avg_distance=('geo_distance_km', 'mean') if 'geo_distance_km' in df.columns else ('amt', 'mean')
    ).round(4)

    # Статистика по времени суток
    hourly_stats = df.groupby('hour').agg(
        transaction_count=('trans_num', 'count'),
        avg_amount=('amt', 'mean'),
        unique_users=('cc_num', 'nunique')
    ).round(4)

    # Статистика по географии (штаты)
    if 'state' in df.columns:
        state_stats = df.groupby('state').agg(
            transaction_count=('trans_num', 'count'),
            avg_amount=('amt', 'mean')
        ).round(4)
    else:
        state_stats = None

    return category_stats, hourly_stats, state_stats

category_stats, hourly_stats, state_stats = compute_basic_stats(df)

print("Топ-10 категорий по количеству транзакций:")
display(category_stats.sort_values('total_transactions', ascending=False).head(10))

print("Активность по часам (пик):")
peak_hour = hourly_stats['transaction_count'].idxmax()
print(f"Пиковый час: {peak_hour}:00 ({hourly_stats.loc[peak_hour, 'transaction_count']:,} транзакций)")

Топ-10 категорий по количеству транзакций:


,total_transactions,unique_users,avg_amount,median_amount,std_amount,avg_distance
category,,,,,,
gas_transport,131659,956,63.4346,62.840,15.8867,85.0745
grocery_pos,123638,979,116.9610,105.120,53.2144,85.0351
home,123115,911,58.2701,48.290,48.6812,84.9718
shopping_pos,116672,961,79.7792,7.760,231.4871,84.9744
kids_pets,113035,914,57.5369,47.180,48.7220,84.9875
shopping_net,97543,977,88.4241,8.440,247.2256,85.0791
entertainment,94014,920,64.2104,50.740,66.1811,84.8057
food_dining,91461,916,51.0869,42.030,48.8542,84.9005
personal_care,90758,918,47.9677,32.585,49.2300,85.0563


Активность по часам (пик):
Пиковый час: 23:00 (67,104 транзакций)


In [31]:
def build_user_profiles(df):
    """
    Строит поведенческие профили для каждого пользователя.
    """
    # Базовые агрегации
    user_profiles = df.groupby('cc_num').agg(
        total_transactions=('trans_num', 'count'),
        total_spent=('amt', 'sum'),
        avg_transaction=('amt', 'mean'),
        std_transaction=('amt', 'std'),
        unique_categories=('category', 'nunique'),
        unique_merchants=('merchant', 'nunique'),
        first_transaction=('trans_date_trans_time', 'min'),
        last_transaction=('trans_date_trans_time', 'max'),
        avg_geo_distance=('geo_distance_km', 'mean') if 'geo_distance_km' in df.columns else ('amt', 'mean')
    ).reset_index()

    # Вычисляем временные метрики
    user_profiles['activity_span_days'] = (
        user_profiles['last_transaction'] - user_profiles['first_transaction']
    ).dt.total_seconds() / 86400

    user_profiles['transactions_per_day'] = (
        user_profiles['total_transactions'] /
        user_profiles['activity_span_days'].replace(0, 1)
    )

    # Временные паттерны: в какое время суток пользователь наиболее активен
    user_hourly = df.groupby(['cc_num', 'hour']).size().reset_index(name='hour_count')
    peak_hours = user_hourly.loc[user_hourly.groupby('cc_num')['hour_count'].idxmax()]
    user_profiles = user_profiles.merge(
        peak_hours[['cc_num', 'hour']].rename(columns={'hour': 'peak_hour'}),
        on='cc_num', how='left'
    )

    # Гео-мобильность: насколько пользователь перемещается
    if 'geo_distance_km' in df.columns:
        user_geo = df.groupby('cc_num')['geo_distance_km'].agg(['mean', 'max', 'std']).reset_index()
        user_geo.columns = ['cc_num', 'avg_distance', 'max_distance', 'std_distance']
        user_profiles = user_profiles.merge(user_geo, on='cc_num', how='left')

    # Заполняем пропуски в std-метриках
    std_cols = [col for col in user_profiles.columns if 'std' in col]
    user_profiles[std_cols] = user_profiles[std_cols].fillna(0)

    print(f"Профили построены для {len(user_profiles):,} пользователей")
    return user_profiles

user_profiles = build_user_profiles(df)
display(user_profiles.head())

Профили построены для 983 пользователей


,cc_num,total_transactions,total_spent,avg_transaction,std_transaction,unique_categories,unique_merchants,first_transaction,last_transaction,avg_geo_distance,activity_span_days,transactions_per_day,peak_hour,avg_distance,max_distance,std_distance
0,60416207185,1518,85043.47,56.023366,122.632635,14,575,2019-01-01 12:47:15,2020-06-21 08:54:21,84.491765,536.838264,2.827667,13,84.491765,155.513075,31.693075
1,60422928733,1531,105640.20,69.000784,102.681962,14,578,2019-01-03 18:38:26,2020-06-21 09:19:28,85.922058,534.611829,2.863760,7,85.922058,153.045236,31.709685
2,60423098130,510,58673.63,115.046333,1202.988005,14,338,2019-01-01 06:48:36,2020-06-19 01:14:31,85.555798,534.767998,0.953685,22,85.555798,153.069235,30.173523
3,60427851591,528,59129.61,111.987898,143.310653,14,358,2019-01-01 07:36:27,2020-06-19 13:06:04,82.778186,535.228900,0.986494,17,82.778186,153.082261,31.589609
4,60487002085,496,25160.11,50.726028,65.843969,14,346,2019-01-06 03:23:55,2020-06-20 15:44:36,85.987411,531.514363,0.933183,19,85.987411,152.194414,30.383657


In [35]:
def identify_interesting_users(user_profiles, df,
                               freq_threshold=5,      # транзакций в час
                               amount_percentile=0.95, # перцентиль для крупных сумм
                               geo_threshold_km=150,   # гео-скачок за 2 часа
                               category_diversity=10): # уникальных категорий
    """
    Выявляет пользователей с необычным поведением на основе поведенческих метрик.
    """
    interesting = set()

    # Критерий: Высокая частота транзакций
    high_freq = user_profiles[
        user_profiles['transactions_per_day'] > freq_threshold * 24  # >5 в час в среднем
    ]['cc_num'].tolist()
    interesting.update(high_freq)
    print(f"- Частые пользователи: {len(high_freq):,}")

    #  Критерий: Высокий средний чек
    amount_threshold = user_profiles['avg_transaction'].quantile(amount_percentile)
    high_amount = user_profiles[
        user_profiles['avg_transaction'] > amount_threshold
    ]['cc_num'].tolist()
    interesting.update(high_amount)
    print(f"- Крупные пользователи (средний чек > ${amount_threshold:.2f}): {len(high_amount):,}")

    # Критерий: Гео-аномалии
    if 'max_distance' in user_profiles.columns:
        geo_mobile = user_profiles[
            user_profiles['max_distance'] > geo_threshold_km
        ]['cc_num'].tolist()
        interesting.update(geo_mobile)
        print(f"- Мобильные пользователи (макс. расстояние > {geo_threshold_km} км): {len(geo_mobile):,}")

    # Критерий: «Разнообразные» — много уникальных категорий
    diverse = user_profiles[
        user_profiles['unique_categories'] >= category_diversity
    ]['cc_num'].tolist()
    interesting.update(diverse)
    print(f"- Разнообразные пользователи (≥{category_diversity} категорий): {len(diverse):,}")

    # Критерий: «Всплески активности» — >5 транзакций за 1 час в любой момент
    df_sorted = df.sort_values(['cc_num', 'trans_date_trans_time'])
    df_sorted['time_diff'] = df_sorted.groupby('cc_num')['trans_date_trans_time'].diff()

    # Находим транзакции, следующие друг за другом с интервалом <10 минут
    rapid_tx = df_sorted[df_sorted['time_diff'] < pd.Timedelta(minutes=10)]
    burst_users = rapid_tx.groupby('cc_num').size()
    burst_users = burst_users[burst_users >= freq_threshold].index.tolist()
    interesting.update(burst_users)
    print(f"- Пользователи со всплесками активности: {len(burst_users):,}")

    interesting_list = list(interesting)
    print(f"\nВсего уникальных интересных пользователей: {len(interesting_list):,}")
    print(f"Доля от всех пользователей: {len(interesting_list)/len(user_profiles)*100:.2f}%")

    return interesting_list

interesting_users = identify_interesting_users(user_profiles, df)

- Частые пользователи: 0
- Крупные пользователи (средний чек > $524.25): 50
- Мобильные пользователи (макс. расстояние > 150 км): 885
- Разнообразные пользователи (≥10 категорий): 908
- Пользователи со всплесками активности: 820

Всего уникальных интересных пользователей: 960
Доля от всех пользователей: 97.66%


In [36]:
interesting_df = df[df['cc_num'].isin(interesting_users)].copy()
normal_df = df[~df['cc_num'].isin(interesting_users)].copy()

print(f"Транзакций интересных пользователей: {len(interesting_df):,}")
print(f"Доля от общего датасета: {len(interesting_df)/len(df)*100:.2f}%")

Транзакций интересных пользователей: 1,296,452
Доля от общего датасета: 99.98%


In [37]:
def compare_segments(normal_df, interesting_df, category_stats):
    """
    Сравнивает поведенческие паттерны двух сегментов пользователей.
    """
    comparison = pd.DataFrame()

    # Сравнение по категориям
    cat_normal = normal_df.groupby('category').agg(
        count=('trans_num', 'count'),
        avg_amount=('amt', 'mean'),
        unique_users=('cc_num', 'nunique')
    ).rename(columns={'count': 'count_normal', 'avg_amount': 'avg_normal', 'unique_users': 'users_normal'})

    cat_interesting = interesting_df.groupby('category').agg(
        count=('trans_num', 'count'),
        avg_amount=('amt', 'mean'),
        unique_users=('cc_num', 'nunique')
    ).rename(columns={'count': 'count_interesting', 'avg_amount': 'avg_interesting', 'unique_users': 'users_interesting'})

    comparison = cat_normal.join(cat_interesting, how='outer').fillna(0)

    # Вычисляем относительные метрики
    comparison['share_normal'] = comparison['count_normal'] / comparison['count_normal'].sum()
    comparison['share_interesting'] = comparison['count_interesting'] / comparison['count_interesting'].sum()
    comparison['share_diff'] = comparison['share_interesting'] - comparison['share_normal']
    comparison['amount_ratio'] = comparison['avg_interesting'] / (comparison['avg_normal'] + 1e-6)

    return comparison.sort_values('share_diff', ascending=False)

segment_comparison = compare_segments(normal_df, interesting_df, category_stats)

print("Топ-10 категорий, где интересные пользователи активнее среднего:")
display(segment_comparison.head(10)[['share_normal', 'share_interesting', 'share_diff', 'amount_ratio']])

Топ-10 категорий, где интересные пользователи активнее среднего:


,share_normal,share_interesting,share_diff,amount_ratio
category,,,,
home,0.000000,0.094963,0.094963,5.827014e+07
kids_pets,0.022422,0.087184,0.064763,2.963156e+00
health_fitness,0.004484,0.066241,0.061756,2.996823e+00
entertainment,0.013453,0.072514,0.059061,1.597920e-01
food_dining,0.031390,0.070542,0.039152,4.137435e-01
personal_care,0.031390,0.070000,0.038609,2.336570e+00
shopping_pos,0.062780,0.089983,0.027202,8.266586e-02
grocery_net,0.008969,0.035057,0.026089,4.978847e+00
misc_pos,0.035874,0.061435,0.025560,8.411442e+00
